In [8]:
import json
with open('output_jierun/answers.jsonl') as f:
    answers = [json.loads(line) for line in f]


In [9]:
annt_path='/home/v-jinjzhao/datasets/jierun/ref_msra_test_coco_o365.json'
with open(annt_path) as f:
    all_annts=json.load(f)

pred_bboxes=[]
gt_bboxes=[]
for idx,annt in enumerate(all_annts):
    pred=answers[idx]
    assert pred['annt_id']==annt['ann_id'], f"{pred['annt_id']}!={annt['ann_id']}"
    pred_bbox=pred['pred_bboxes'][0]
    gt_bbox=annt['bbox']
    x,y,w,h=gt_bbox
    x1,y1,x2,y2=[x,y,x+w,y+h]
    pred_bboxes.append(pred_bbox)
    gt_bboxes.append([x1,y1,x2,y2])



In [10]:
import json
import torch
from tqdm import tqdm
from overlaps import bbox_overlaps
# import average from math
from statistics import mean


def calculate_iou_acc(pred_bboxes,gt_bboxes, thresh=0.5):
    """
    pred_bboxes: [N,4]
    gt_bboxes: [N,4]
    calculate the iou and acc of the pred_bboxes and gt_bboxes,
    if iou(pred_bboxes[i],gt_bboxes[i])>0.5, then acc+=1
    all pred_bboxes_i and gt_bboxes_i are one to one assigned.
    
    """
    iou=bbox_overlaps(pred_bboxes,gt_bboxes,mode='iou', is_aligned=True)
    if(type(thresh) is not list):
        thresh=[thresh]
    accs=dict()
    for t in thresh:
        accs[t]=(iou>t).sum().item()/len(iou)
    return iou,accs




pred_bboxes=torch.tensor(pred_bboxes)
gt_bboxes=torch.tensor(gt_bboxes)
# create a list fron 0.5 to 0.9 with step 0.05
thresh=[i/100 for i in range(50,95,5)]
print(thresh)
iou,acc=calculate_iou_acc(pred_bboxes,gt_bboxes,thresh)
print_ths=[0.5,0.7,0.9]
for k,v in acc.items():
    if(k in print_ths):
        print(f"thresh={k}, acc={v}")
print(f"iou|0.5:0.9={mean([v for v in acc.values()])}")

print(iou)
print(acc)
print(f"{len(iou)}/{len(all_annts)}")

[0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
thresh=0.5, acc=0.6083677025208972
thresh=0.7, acc=0.46189982576476035
thresh=0.9, acc=0.12000176440748991
iou|0.5:0.9=0.4213797176457903
tensor([0.0000, 0.0478, 0.8290,  ..., 0.0402, 0.4664, 0.0000])
{0.5: 0.6083677025208972, 0.55: 0.5770494695749984, 0.6: 0.5453342449438698, 0.65: 0.506230563948744, 0.7: 0.46189982576476035, 0.75: 0.4047771332789308, 0.8: 0.33245848128625305, 0.85: 0.23629827308616924, 0.9: 0.12000176440748991}
45341/45341


In [11]:
for idx,annt in enumerate(all_annts):
    annt['groundingGPT_pred']=dict(
        pred_bboxes=pred_bboxes[idx].tolist(),
        iou=iou[idx].item()
    )


In [12]:
with open(annt_path,'w') as f:
    json.dump(all_annts,f,indent=4)

In [15]:
num_shikra=0
num_groundGPT=0
for annt in all_annts:
    if('shikra7b_pred' in annt):
        num_shikra+=1
    if('groundingGPT_pred' in annt):
        num_groundGPT+=1
print(f"num_shikra={num_shikra}, num_groundGPT={num_groundGPT}")


num_shikra=41218, num_groundGPT=44738
